# ZADANIE: Model Card Tensor

- przygotuj dataframe w oparciu o specyfikacje "model cards" dla poszczególnych modeli

# DOCS

- [dokumentacja pliku HF:`config.json`](https://huggingface.co/docs/transformers/main_classes/configuration)
- model cards:
  1. [Bielik-7B-v0.1](https://huggingface.co/speakleash/Bielik-7B-v0.1)
  2. [Llama-3.1-8B](https://huggingface.co/meta-llama/Llama-3.1-8B)
  3. [Mistral-7B-v0.1](https://huggingface.co/mistralai/Mistral-7B-v0.1)
  4. dla ambitnych 🔥 (inna struktura)
    - [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1)
    - [Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B)

In [1]:
!pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
from pathlib import Path
import json
import sys
import os
import pandas as pd

base = Path(os.getcwd())
pattern = "*-config.json"
matches = sorted(base.rglob(pattern))
files = [p.name for p in matches]
# print(files)
# print(json.dumps(files, indent=2))

failing = []
model_cards = []

for p in matches:
    try:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        model_cards.append({
            'filename': p.name,
            'json': data
        })
    except json.JSONDecodeError:
        failing.append(f"Niepoprawny format JSON (Pusty/Błędny) w pliku: {p.name}")
    except ValueError as e:
        failing.append(f"Błąd danych: {e} Plik: {p.name}")
    except Exception as e:
        failing.append(f"Inny nieznany błąd przy wczytywaniu {p.name}: {e}")

if len(failing):
    print(failing)
else:
    print('All models calrds loaded successfully')

df = pd.DataFrame(model_cards)
df['model_type'] = df['json'].apply(lambda x: x.get('model_type', None))
df_wynikowy = df[['filename', 'model_type']]

display(df_wynikowy)



All models calrds loaded successfully


,filename,model_type
0,Bielik-7B-Instruct-v0.1-config.json,mistral
1,DeepSeek-R1-config.json,deepseek_v3
2,Llama-3.1-8B-config.json,llama
3,Mistral-7B-v0.1-config.json,mistral
4,Qwen2.5-7B-Instruct-config.json,qwen2


In [5]:
import pandas as pd

def generate_blank(c):
    return ['[?, ?]' for _ in range(c)]

data = {
    'Bielik-7B-Instruct-v0.1': generate_blank(10),
    'Llama-3.1-8B': generate_blank(10),
    'Mistral-7B-v0.1': generate_blank(10),
    # 'DeepSeek-R1': generate_blank(10),
    # 'Qwen2.5-7B': generate_blank(10),
}

tensors = [
    'embed_tokens.weight',
    'input_layernorm.weight',
    'mlp.down_proj.weight',
    'mpl.gate_proj.weight',
    'mpl.up_proj.weight',
    'post_attention_layernorm.weight',
    'self_attn.k_proj.weight',
    'self_attn.o_proj.weight',
    'self_attn.q_proj.weight',
    'self_attn.v_proj.weight',
]

df = pd.DataFrame(data, index=tensors)

display(df)
# display(df.T) # transpozycja (obrócenie)


# Enhancement: replace blank tensor shapes with inferred ones based on loaded configs

# Reuse already loaded model_cards list (filename + json)
# Build a dict keyed by base model name (strip suffix -config.json if present earlier)
configs = {}
for row in model_cards:
    # Derive a simple key without trailing '-config.json'
    name = row['filename'].replace('-config.json', '')
    # Optionally remove file extension variants already handled:
    name = name.replace('.json', '')
    configs[name] = row['json']

# Correct tensor names (typo: mpl -> mlp) and canonical ordering
tensors = [
    'embed_tokens.weight',
    'input_layernorm.weight',
    'post_attention_layernorm.weight',
    'mlp.gate_proj.weight',
    'mlp.up_proj.weight',
    'mlp.down_proj.weight',
    'self_attn.q_proj.weight',
    'self_attn.k_proj.weight',
    'self_attn.v_proj.weight',
    'self_attn.o_proj.weight',
]

def fmt(shape):
    return '[' + ', '.join(str(x) for x in shape) + ']'

def infer_shapes(cfg):
    hidden_size = cfg.get('hidden_size')
    intermediate_size = cfg.get('intermediate_size')
    vocab_size = cfg.get('vocab_size')
    num_heads = cfg.get('num_attention_heads')
    kv_heads = cfg.get('num_key_value_heads', num_heads)
    # Basic validation
    if not all(isinstance(v, int) and v > 0 for v in [hidden_size, intermediate_size, vocab_size, num_heads, kv_heads]):
        return ['[?, ?]'] * len(tensors)
    head_dim = hidden_size // num_heads if num_heads else None
    if head_dim is None or hidden_size % num_heads != 0:
        return ['[?, ?]'] * len(tensors)
    shapes = {
        'embed_tokens.weight': (vocab_size, hidden_size),
        'input_layernorm.weight': (hidden_size,),
        'post_attention_layernorm.weight': (hidden_size,),
        'mlp.gate_proj.weight': (intermediate_size, hidden_size),
        'mlp.up_proj.weight': (intermediate_size, hidden_size),
        'mlp.down_proj.weight': (hidden_size, intermediate_size),
        'self_attn.q_proj.weight': (hidden_size, hidden_size),
        'self_attn.k_proj.weight': (hidden_size, kv_heads * head_dim),
        'self_attn.v_proj.weight': (hidden_size, kv_heads * head_dim),
        'self_attn.o_proj.weight': (hidden_size, hidden_size),
    }
    return [fmt(shapes[t]) for t in tensors]

# Assemble DataFrame with inferred shapes
shape_table = {}
for model_name, cfg in configs.items():
    shape_table[model_name] = infer_shapes(cfg)

shapes_df = pd.DataFrame(shape_table, index=tensors)

display(shapes_df)

,Bielik-7B-Instruct-v0.1,Llama-3.1-8B,Mistral-7B-v0.1
embed_tokens.weight,"[?, ?]","[?, ?]","[?, ?]"
input_layernorm.weight,"[?, ?]","[?, ?]","[?, ?]"
mlp.down_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
mpl.gate_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
mpl.up_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
post_attention_layernorm.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.k_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.o_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.q_proj.weight,"[?, ?]","[?, ?]","[?, ?]"
self_attn.v_proj.weight,"[?, ?]","[?, ?]","[?, ?]"


,Bielik-7B-Instruct-v0.1,DeepSeek-R1,Llama-3.1-8B,Mistral-7B-v0.1,Qwen2.5-7B-Instruct
embed_tokens.weight,"[32000, 4096]","[129280, 7168]","[128256, 4096]","[32000, 4096]","[152064, 3584]"
input_layernorm.weight,[4096],[7168],[4096],[4096],[3584]
post_attention_layernorm.weight,[4096],[7168],[4096],[4096],[3584]
mlp.gate_proj.weight,"[14336, 4096]","[18432, 7168]","[14336, 4096]","[14336, 4096]","[18944, 3584]"
mlp.up_proj.weight,"[14336, 4096]","[18432, 7168]","[14336, 4096]","[14336, 4096]","[18944, 3584]"
mlp.down_proj.weight,"[4096, 14336]","[7168, 18432]","[4096, 14336]","[4096, 14336]","[3584, 18944]"
self_attn.q_proj.weight,"[4096, 4096]","[7168, 7168]","[4096, 4096]","[4096, 4096]","[3584, 3584]"
self_attn.k_proj.weight,"[4096, 1024]","[7168, 7168]","[4096, 1024]","[4096, 1024]","[3584, 512]"
self_attn.v_proj.weight,"[4096, 1024]","[7168, 7168]","[4096, 1024]","[4096, 1024]","[3584, 512]"
self_attn.o_proj.weight,"[4096, 4096]","[7168, 7168]","[4096, 4096]","[4096, 4096]","[3584, 3584]"
